# Chapter 4: Action Representations
Three ways to emit an action -- discrete tokens, MSE regression, DDPM diffusion -- each with chunk size K=1 vs K=4.

In [ ]:
!pip install torch torchvision numpy gymnasium matplotlib transformers

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/04_action_representations

## Continuous Actions

Chapters 1-3 used 5 discrete actions. Real robots take continuous ones, so the task moves to `ContinuousMiniPushT`: actions are `(dx, dy)` in `[-1, 1]`.

That single change is what makes "how do we represent an action?" a real question.

In [ ]:
from mini_pusht_continuous import ContinuousMiniPushT, ContinuousExpert
import numpy as np
import matplotlib.pyplot as plt

env = ContinuousMiniPushT(size=224)
expert = ContinuousExpert(size=224, noise_std=0.1)

obs, info = env.reset(seed=0)
print(f"Action space: {env.action_space}")
print(f"Example expert action: {expert.act(info)}")

plt.figure(figsize=(4, 4))
plt.imshow(obs)
plt.title("ContinuousMiniPushT -- push block to goal")
plt.axis("off")
plt.show()

## Representation 1: Discretize

RT-2 and OpenVLA turn each action dimension into one of **256 bins**, then predict a bin like a language token. It's lossless enough in practice -- but let's see the quantization error directly.

In [ ]:
import torch
from action_heads import continuous_to_bins, bins_to_continuous, N_BINS

actions = torch.tensor([[0.0, 1.0], [-1.0, 0.5], [0.137, -0.42]])
bins = continuous_to_bins(actions, n_bins=N_BINS)
recovered = bins_to_continuous(bins, n_bins=N_BINS)

print(f"n_bins = {N_BINS}\n")
for a, b, r in zip(actions, bins, recovered):
    print(f"  {a.tolist()}  ->  bin {b.tolist()}  ->  {[round(v, 4) for v in r.tolist()]}")

print(f"\nMax round-trip error: {(actions - recovered).abs().max():.5f}")
print(f"Bin width: {2.0 / N_BINS:.5f}")

## Collect Demos

Written straight to disk as a memory-mapped array. 1000 episodes of 224x224 frames is several GB -- holding it in RAM is how Colab sessions die.

In [ ]:
import os
from action_heads import collect_demos_to_disk, MmapPushTDataset, PRESETS

preset = PRESETS["quick"]     # {"n_demos": 1000, "epochs": 15, "batch_size": 128}
print(f"preset: {preset}")

os.makedirs("checkpoints", exist_ok=True)
demo_path = f"checkpoints/demos_224x224_{preset['n_demos']}ep.npz"

if not os.path.exists(demo_path):
    total = collect_demos_to_disk(env, expert, demo_path, n_episodes=preset["n_demos"])
    print(f"Collected {total} transitions -> {demo_path}")
else:
    print(f"Reusing {demo_path}")

## Precompute SigLIP Embeddings

The vision encoder is frozen, so its output never changes across training runs. Encoding once and reusing the embeddings for all six configs turns six SigLIP passes into one.

We do it per chunk size, because K changes how actions are grouped (not how images are encoded).

In [ ]:
from action_heads import VisionEncoder, precompute_embeddings, CHUNK_SIZES

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

vision_encoder = VisionEncoder()
embed_datasets = {}
for cs in CHUNK_SIZES:
    raw = MmapPushTDataset(demo_path, chunk_size=cs)
    embed_datasets[cs] = precompute_embeddings(vision_encoder, raw, device=device, batch_size=64)
    print(f"  chunk_size={cs}: {len(embed_datasets[cs])} samples")

vision_encoder = vision_encoder.cpu()
if device != "cpu":
    torch.cuda.empty_cache()

## Train All Six Configurations

3 head types x 2 chunk sizes. Only the head trains -- vision stays frozen -- so each run is fast.

In [ ]:
import time
from action_heads import build_vla, train, evaluate, HEAD_TYPES, config_name

results = {}
losses_dict = {}

for head in HEAD_TYPES:
    for cs in CHUNK_SIZES:
        name = config_name(head, cs)
        print(f"\n--- {name} ---")

        model = build_vla(head, chunk_size=cs)
        t0 = time.time()
        losses = train(model, embed_datasets[cs], epochs=preset["epochs"],
                       batch_size=preset["batch_size"], device=device)
        elapsed = time.time() - t0

        metrics = evaluate(model, env, vision_encoder, n_episodes=100, device=device)
        metrics["time"] = elapsed
        results[name] = metrics
        losses_dict[name] = losses

        print(f"  success={metrics['success_rate']*100:.0f}%  "
              f"smoothness={metrics['mean_smoothness']:.3f}  "
              f"length={metrics['mean_length']:.0f}  ({elapsed:.0f}s)")

## Results

In [ ]:
print(f"{'Config':<16} {'Success':>8} {'Smoothness':>11} {'Avg Length':>11}")
print("-" * 50)
for name, r in results.items():
    print(f"{name:<16} {r['success_rate']*100:>7.0f}% {r['mean_smoothness']:>11.3f} {r['mean_length']:>11.0f}")

names = list(results.keys())
x = np.arange(len(names))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(x, [results[n]["success_rate"] * 100 for n in names])
axes[0].set_ylabel("Success rate (%)")
axes[0].set_title("Success by action representation")

axes[1].bar(x, [results[n]["mean_smoothness"] for n in names], color="tab:orange")
axes[1].set_ylabel("Mean smoothness (lower = smoother)")
axes[1].set_title("Trajectory smoothness")

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## The Chunking Effect

Read the bars in pairs. Every head type improves when it predicts 4 actions at once instead of 1 -- and the trajectories get markedly smoother, because consecutive actions come from a single coherent prediction rather than 4 independent ones.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
w = 0.35
heads = HEAD_TYPES
xh = np.arange(len(heads))

for i, cs in enumerate(CHUNK_SIZES):
    vals = [results[config_name(h, cs)]["success_rate"] * 100 for h in heads]
    ax.bar(xh + (i - 0.5) * w, vals, w, label=f"K={cs}")

ax.set_xticks(xh)
ax.set_xticklabels([h.capitalize() for h in heads])
ax.set_ylabel("Success rate (%)")
ax.set_title("Action chunking: K=1 vs K=4")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, losses in losses_dict.items():
    ax.plot(range(1, len(losses) + 1), losses, marker="o", markersize=2, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_yscale("log")
ax.set_title("Training loss by action head (log scale)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## What We Learned

**Action chunking is the biggest lever.** K=4 roughly doubles success for every head type. Predicting a short sequence gives the policy commitment; predicting one step at a time lets it dither.

**Regression wins on this task** -- 95% with K=4 at full scale. MiniPushT is unimodal: from any state there is essentially one right way to push the block. MSE regression is exactly the right tool for that, and it is the cheapest of the three.

**Diffusion fails here (~0%), and that is the instructive part.** A 2-layer MLP denoiser with 10 inference steps has nowhere near the capacity to learn a useful score function, and the task has no multi-modality for diffusion to exploit. This is not evidence that diffusion is wrong for VLAs -- it is evidence that diffusion needs *scale*. Chapter 6 rebuilds it as a ~12M-param transformer with flow matching, and it works.

**Discrete stays competitive** at 82% (K=4). Quantization to 256 bins costs less than you would guess -- the bin width is 0.0078, well below the precision the task needs.

**Next:** Chapter 5 drops the toy environment and loads real robot data (PushT, ALOHA) through LeRobot.